# This noteboook runs Monte Carlo simulations of the estimator

## Setup

In [3]:
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

from Chorms import *

do_compile = True # should the c-functions be re-compiled?

import os
os.environ.pop('NoDefaultCurrentDirectoryInExePath', None)

do_reinstall_nlopt = False # If problems with NLOPT during re-compilation of c++ files, then try re-installing NLOPT by swithcing this to True and delete the folder "nlopt-2.4.2-dll64" in the cppfuncs folder before running this notebook.
if do_reinstall_nlopt:
    from EconModel import cpptools
    cpptools.setup_nlopt(folder='cppfuncs/', do_print=False,download=False,unzip=True)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Setup Model

In [4]:
model = ChormsClass(par={'do_gridsearch':False,'reuse_init':True})
model.link_to_cpp(force_compile=do_compile)

In [5]:
# parameters to estime. Start will all the MLE-related parameters
# theta_names = [*model.params_wage,*model.params_meas]
theta_names = model.params_wage
theta_init = np.array([getattr(model.par, name) for name in theta_names]) * 0.95

In [6]:
# load/simulate true data (if seed is changed before this call, the data will change)
model.load_data(do_simulate=True)
theta_true = np.array([getattr(model.par, name) for name in theta_names])

In [7]:
# initial guess for wage parameters
initial_wage = initial_wage_parameters(model)

## Test estimator

In [8]:
print('True parameters:')
[print(f'{name}={value:.3f}',end=' ') for name, value in zip(theta_names, theta_true)];

print('\nInitial wage parameters:')
[print(f'{name}={value:.3f}',end=' ') for name, value in zip(['wage_logmean_w', 'wage_logmean_m', 'wage_chol1', 'wage_chol2', 'wage_chol3'], initial_wage)];

True parameters:
wage_logmean_w=1.000 wage_logmean_m=1.000 wage_chol1=-0.700 wage_chol2=0.250 wage_chol3=0.000 
Initial wage parameters:
wage_logmean_w=1.016 wage_logmean_m=1.015 wage_chol1=-0.730 wage_chol2=0.319 wage_chol3=-0.155 

In [9]:
obj_func = lambda theta: obj_loglik(theta,theta_names,model,do_print=True)
res = minimize(obj_func,theta_init,method='Nelder-Mead',options={'maxiter':25}) # note maxiter

wage_logmean_w=0.950 wage_logmean_m=0.950 wage_chol1=-0.665 wage_chol2=0.237 wage_chol3=0.000 ->obj: -1.7037
wage_logmean_w=0.997 wage_logmean_m=0.950 wage_chol1=-0.665 wage_chol2=0.237 wage_chol3=0.000 ->obj: -1.7128
wage_logmean_w=0.950 wage_logmean_m=0.997 wage_chol1=-0.665 wage_chol2=0.237 wage_chol3=0.000 ->obj: -1.7009
wage_logmean_w=0.950 wage_logmean_m=0.950 wage_chol1=-0.698 wage_chol2=0.237 wage_chol3=0.000 ->obj: -1.7060
wage_logmean_w=0.950 wage_logmean_m=0.950 wage_chol1=-0.665 wage_chol2=0.249 wage_chol3=0.000 ->obj: -1.7043
wage_logmean_w=0.950 wage_logmean_m=0.950 wage_chol1=-0.665 wage_chol2=0.237 wage_chol3=0.000 ->obj: -1.7037
wage_logmean_w=0.969 wage_logmean_m=0.902 wage_chol1=-0.678 wage_chol2=0.242 wage_chol3=0.000 ->obj: -1.7099
wage_logmean_w=0.977 wage_logmean_m=0.931 wage_chol1=-0.684 wage_chol2=0.244 wage_chol3=0.000 ->obj: -1.7128
wage_logmean_w=0.990 wage_logmean_m=0.922 wage_chol1=-0.693 wage_chol2=0.247 wage_chol3=0.000 ->obj: -1.7163
wage_logmean_w=0.99

In [10]:
# illustrate the profiling.
print(model.par.meas_sigma_labor_w)
print(model.par.meas_sigma_vec)

0.1
[0.0990819  0.09900934 0.09961758 0.09976094 0.09908636 0.09225859]


In [11]:
# Very slow! This is because the numerical gradients uses central finte differences and thus do not re-use any information. Requires 2*K+4*K^4 evaluations of the objective function. For K=5 this is 4*5^2+2*5=110.
# This could be changed to only forward differences and then the baseline likelihood can be forwarded in all numerical gradients. 
%time se = standard_errors_mle(res.x, theta_names, model, clusters=None)
print(se)

CPU times: total: 58min 50s
Wall time: 3min 48s
[0.00085401 0.00141652 0.0015616  0.00183007 0.00107529]
